In [ ]:
!pip install mediapipe==0.10.21

In [ ]:
static_image_mode=False


In [ ]:
import cv2 # Accessing webcam or showing video window
import mediapipe as mp # Used for FaceMesh(face landmarks + iris detection)
import numpy as np

In [ ]:
mp_face_mesh=mp.solutions.face_mesh #Accesses MediaPipe's FaceMesh Module

## ***# These are landMark IDs provided by Mediapipe. They represent points around the iris(pupil area)***

In [ ]:
Left_IRIS =[474,475,476,477]
Right_IRIS=[469,470,471,472]

**These are eye boundary landmarks.
They represent the left corner and right corner of the eye.**

In [ ]:
LEFT_EYE=[33,133]
RIGHT_EYE=[362,263]

**This is used to measure how far iris is from the center**

In [ ]:
def euclidean_distance(p1,p2):
  return np.linalg.norm(np.array(p1)-np.array(p2))

In [ ]:
!pip install mediapipe==0.10.21 --force-reinstall

  Using cached mediapipe-0.10.21-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (9.7 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached jax-0.9.0.1-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.9.0.1-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (1.3 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached protobuf-4.25.8-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
  Using cached sounddevice-0.5.5-py3-none-any.whl.metadata (1.4 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
from base64 import b64decode
import matplotlib.pyplot as plt

mp_face_mesh = mp.solutions.face_mesh

# Iris landmarks indexes (MediaPipe)
LEFT_IRIS = [474, 475, 476, 477]
RIGHT_IRIS = [469, 470, 471, 472]

# Eye boundary landmarks (approx)
LEFT_EYE = [33, 133]
RIGHT_EYE = [362, 263]

def euclidean_distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

# Webcam capture function (Colab JS)
def capture_frame():
    js = Javascript('''
        async function captureFrame() {
          const video = document.createElement('video');
          video.style.display = 'block';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          document.body.appendChild(video);
          video.srcObject = stream;
          await video.play();

          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);

          stream.getVideoTracks()[0].stop();
          video.remove();

          return canvas.toDataURL('image/jpeg', 0.8);
        }
    ''')
    display(js)
    data = eval_js('captureFrame()')
    binary = b64decode(data.split(',')[1])

    np_arr = np.frombuffer(binary, np.uint8)
    frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    return frame


# FaceMesh Setup
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    refine_landmarks=True,
    max_num_faces=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

print("✅ Starting Live Eye Tracking in Colab (Press stop button to end)")

while True:
    frame = capture_frame()
    frame = cv2.flip(frame, 1)

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w, _ = frame.shape

    results = face_mesh.process(rgb_frame)

    confidence = 0
    status = "Face Not Detected"
    color = (255, 0, 0)

    if results.multi_face_landmarks:
        mesh_points = np.array(
            [(int(p.x * w), int(p.y * h)) for p in results.multi_face_landmarks[0].landmark]
        )

        # LEFT EYE
        left_iris_center = np.mean(mesh_points[LEFT_IRIS], axis=0).astype(int)
        left_eye_left = mesh_points[LEFT_EYE[0]]
        left_eye_right = mesh_points[LEFT_EYE[1]]

        # RIGHT EYE
        right_iris_center = np.mean(mesh_points[RIGHT_IRIS], axis=0).astype(int)
        right_eye_left = mesh_points[RIGHT_EYE[0]]
        right_eye_right = mesh_points[RIGHT_EYE[1]]

        # Draw iris centers
        cv2.circle(frame, tuple(left_iris_center), 3, (0, 255, 0), -1)
        cv2.circle(frame, tuple(right_iris_center), 3, (0, 255, 0), -1)

        # Eye midpoints
        left_eye_mid = ((left_eye_left + left_eye_right) / 2).astype(int)
        right_eye_mid = ((right_eye_left + right_eye_right) / 2).astype(int)

        # Distances
        left_dist = euclidean_distance(left_iris_center, left_eye_mid)
        right_dist = euclidean_distance(right_iris_center, right_eye_mid)

        # Eye width
        left_eye_width = euclidean_distance(left_eye_left, left_eye_right)
        right_eye_width = euclidean_distance(right_eye_left, right_eye_right)

        # Normalized gaze score
        left_score = left_dist / left_eye_width
        right_score = right_dist / right_eye_width

        gaze_score = (left_score + right_score) / 2

        # Confidence
        confidence = max(0, 100 - (gaze_score * 300))

        # Status
        if confidence > 70:
            status = "Focused on Interviewer"
            color = (0, 255, 0)
        elif confidence > 40:
            status = "Partially Focused"
            color = (0, 255, 255)
        else:
            status = "Not Focused"
            color = (0, 0, 255)

        # Display text
        cv2.putText(frame, f"Status: {status}", (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

        cv2.putText(frame, f"Confidence: {confidence:.2f}%", (30, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    # Convert BGR to RGB for matplotlib
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    clear_output(wait=True)
    plt.figure(figsize=(8,6))
    plt.imshow(frame_rgb)
    plt.axis("off")
    plt.show()

    print(f"🎯 Status: {status}")
    print(f"📌 Confidence: {confidence:.2f}%")